<h2> Loading Libraries

In [1]:
library(Seurat)
library(CellChat)
library(patchwork)
library(dplyr)
library(ggplot2)
library(NMF)
library(ggalluvial)
options(stringsAsFactors = FALSE)
library(ComplexHeatmap)
library(grid)
library(foreach)

Warning message:
"pakiet 'Seurat' został zbudowany w wersji R 4.4.3"
Ładowanie wymaganego pakietu: SeuratObject

Warning message:
"pakiet 'SeuratObject' został zbudowany w wersji R 4.4.3"
Ładowanie wymaganego pakietu: sp

Warning message:
"pakiet 'sp' został zbudowany w wersji R 4.4.3"

Dołączanie pakietu: 'SeuratObject'


Następujące obiekty zostały zakryte z 'package:base':

    intersect, t


Ładowanie wymaganego pakietu: dplyr

Warning message:
"pakiet 'dplyr' został zbudowany w wersji R 4.4.3"

Dołączanie pakietu: 'dplyr'


Następujące obiekty zostały zakryte z 'package:stats':

    filter, lag


Następujące obiekty zostały zakryte z 'package:base':

    intersect, setdiff, setequal, union


Ładowanie wymaganego pakietu: igraph

Warning message:
"pakiet 'igraph' został zbudowany w wersji R 4.4.3"

Dołączanie pakietu: 'igraph'


Następujące obiekty zostały zakryte z 'package:dplyr':

    as_data_frame, groups, union


Następujący obiekt został zakryty z 'package:Seurat':

    compo

<h4> Wczytanie zintegrowanego i annotowanego obiektu

In [2]:
seurat_obj <- readRDS("SCT_annotated_KSz.rds")


In [3]:
DefaultAssay(seurat_obj) <- "RNA"

In [ ]:
seurat_obj <- NormalizeData(seurat_obj, normalization.method= "LogNormalize", scale.factor = 10000)

Normalizing layer: counts



In [5]:
seurat_obj$samples <- seurat_obj$patient
Idents(seurat_obj) <- "cluster_annotation"

<h3> Usunięcie blastów

In [6]:
leukemia_clusters <- c(
  "Blasts_transB", "Blasts_preB_transB_4", "Blasts_preB_memoryB",
  "PreB_mixed_progenitor", "Blasts_preB_transB_2", 
  "ProB_CLP_progenitor", "Blasts_preB_mix_LMPP_erythro", "Blasts_PreB", 
  "Blasts_preB_transB", "Blasts_preB_transB_3", "Blasts_preB_transB_CLP", 
  "Blasts_preB"
)

In [7]:
seurat_micro <- subset(seurat_obj, idents = leukemia_clusters, invert = TRUE)

In [8]:
seurat_ctrl <- subset(seurat_micro, subset = sample_type == "cntrl")
seurat_leuk <- subset(seurat_micro, subset = sample_type != "cntrl")

In [14]:
seurat_ctrl_blast <- subset(seurat_obj, subset = sample_type == "cntrl")
seurat_leuk_blast <- subset(seurat_obj, subset = sample_type != "cntrl")

In [9]:
run_cellchat <- function(seurat_data) {
  chat <- createCellChat(object = seurat_data, group.by = "cluster_annotation", assay = "RNA")
  chat@DB <- CellChatDB.human
  chat <- subsetData(chat) %>% 
    identifyOverExpressedGenes() %>% 
    identifyOverExpressedInteractions() %>% 
    computeCommunProb(type = "triMean", nboot = 100) %>% 
    filterCommunication(min.cells = 10) %>% 
    computeCommunProbPathway() %>% 
    aggregateNet() %>% 
    netAnalysis_computeCentrality(slot.name = "netP")
  return(chat)
}

In [10]:
cellchat_ctrl <- run_cellchat(seurat_ctrl)
cellchat_leuk <- run_cellchat(seurat_leuk)

[1] "Create a CellChat object from a Seurat object"


Warning message:
"The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.
ℹ The deprecated feature was likely used in the CellChat package.
  Please report the issue to the authors."


The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  Abnormal _progenitors BaEoMa_progenitor CD14_monocytes CD4_Tcells CD4_Tcells_2 Diff_erythro Early_erythro Erythro_megakaro_progenitor HSC HSC_2 HSC_abberant HSC_abberant_2 HSC_abberant_3 HSC_abberant_4  Late_erythro Late_erythro_2 Late_erythro_3 Mono_macrophage NK pDC pre_pDC_cDC2_GMP PreB PreB_erythro_prog TransitionalB_memoryB 
triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2026-05-26 23:13:29.993232]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2026-05-26 23:21:43.270373]"
[1] "Create a CellChat object from a Seurat object"
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat a

In [15]:
cellchat_ctrl_blast <- run_cellchat(seurat_ctrl_blast)
cellchat_leuk_blast <- run_cellchat(seurat_leuk_blast)

[1] "Create a CellChat object from a Seurat object"
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  Abnormal _progenitors BaEoMa_progenitor Blasts_preB Blasts_PreB Blasts_preB_memoryB Blasts_preB_mix_LMPP_erythro Blasts_preB_transB Blasts_preB_transB_2 Blasts_preB_transB_3 Blasts_preB_transB_4 Blasts_preB_transB_CLP Blasts_transB CD14_monocytes CD4_Tcells CD4_Tcells_2 Diff_erythro Early_erythro Erythro_megakaro_progenitor HSC HSC_2 HSC_abberant HSC_abberant_2 HSC_abberant_3 HSC_abberant_4  Late_erythro Late_erythro_2 Late_erythro_3 Mono_macrophage NK pDC pre_pDC_cDC2_GMP PreB PreB_erythro_prog PreB_mixed_progenitor ProB_CLP_progenitor TransitionalB_memoryB 
triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2026-05-26 23:50:46.638007]"
[1] ">>> CellChat inference is done. Parameter values a

In [11]:
object.list <- list(CTRL = cellchat_ctrl, LEUK = cellchat_leuk)
cellchat_merged <- mergeCellChat(object.list, add.names = names(object.list))

Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



In [16]:
object.list1 <- list(CTRL = cellchat_ctrl_blast, LEUK = cellchat_leuk_blast)
cellchat_merged1 <- mergeCellChat(object.list1, add.names = names(object.list1))

Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



In [13]:
saveRDS(cellchat_ctrl, file="cellchat_bez_blastow_ctrl.rds")
saveRDS(cellchat_leuk, file="cellchat_bez_blastow_leuk.rds")
saveRDS(cellchat_merged, file="cellchat_bez_blastow_merged.rds")

In [17]:
saveRDS(cellchat_ctrl_blast, file="cellchat_ctrl_blast.rds")
saveRDS(cellchat_leuk_blast, file="cellchat_leuk_blast.rds")
saveRDS(cellchat_merged1, file="cellchat_merged_blast.rds")